# Building an Image Classifier with CNN using MNIST Dataset

**Objective:** Build a Convolutional Neural Network (CNN) to classify handwritten digits from 0 to 9 using the MNIST dataset.

The notebook covers:
- Loading the MNIST dataset
- Data preprocessing
- CNN model creation
- Model training and validation
- Test-set evaluation
- Training/validation plots
- Sample predictions
- Saving the trained model
- Testing custom handwritten digit images

## Step 1: Import Libraries and Load Dataset

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
from PIL import Image
import os

print("TensorFlow version:", tf.__version__)

(x_train, y_train), (x_test, y_test) = mnist.load_data()

print("Training images:", x_train.shape)
print("Training labels:", y_train.shape)
print("Testing images:", x_test.shape)
print("Testing labels:", y_test.shape)

### Dataset Information

MNIST contains 60,000 training images and 10,000 testing images. Each image is a 28 × 28 grayscale image representing a handwritten digit from 0 to 9.

## Step 2: Preprocess the Data

In [ ]:
# Normalize pixel values from 0-255 to 0-1
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Reshape images to 28x28x1 for CNN input
x_train = x_train.reshape(-1, 28, 28, 1)
x_test = x_test.reshape(-1, 28, 28, 1)

# Convert labels to one-hot encoded format
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

print("Training image shape:", x_train.shape)
print("Testing image shape:", x_test.shape)
print("Training label shape:", y_train.shape)
print("Testing label shape:", y_test.shape)

### Display Sample MNIST Images

In [ ]:
plt.figure(figsize=(10, 4))

for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i].reshape(28, 28), cmap="gray")
    plt.title(f"Digit: {np.argmax(y_train[i])}")
    plt.axis("off")

plt.tight_layout()
plt.show()

## Step 3: Build the CNN Model

In [ ]:
model = models.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(10, activation="softmax")
])

model.summary()

### Model Architecture

- **Conv2D:** Extracts visual features such as edges, curves, and shapes.
- **MaxPooling2D:** Reduces the spatial dimensions while retaining important features.
- **Dropout:** Helps reduce overfitting.
- **Flatten:** Converts feature maps into a one-dimensional vector.
- **Dense:** Learns the final classification features.
- **Softmax:** Produces probabilities for the 10 digit classes.

## Step 4: Compile and Train the Model

In [ ]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    x_train,
    y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.1
)

## Step 5: Evaluate the Model

In [ ]:
test_loss, test_accuracy = model.evaluate(
    x_test,
    y_test,
    verbose=2
)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

if test_accuracy >= 0.98:
    print("The model achieved the required accuracy of at least 98%.")
else:
    print("The model did not reach 98% accuracy in this run.")

### Training and Validation Accuracy

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.legend()
plt.show()

### Training and Validation Loss

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.show()

## Sample Predictions on Test Images

In [ ]:
predictions = model.predict(x_test[:10], verbose=0)

plt.figure(figsize=(10, 4))

for i in range(10):
    predicted_digit = np.argmax(predictions[i])
    actual_digit = np.argmax(y_test[i])

    plt.subplot(2, 5, i + 1)
    plt.imshow(x_test[i].reshape(28, 28), cmap="gray")
    plt.title(f"Actual: {actual_digit}\nPredicted: {predicted_digit}")
    plt.axis("off")

plt.tight_layout()
plt.show()

## Step 6: Save the Trained Model

In [ ]:
model.save("mnist_cnn_model.keras")
print("Model saved successfully as mnist_cnn_model.keras")

## Step 7: Test the Model with Custom Images

Place your handwritten digit image in the same folder as this notebook.

The function below:
1. Opens the image
2. Converts it to grayscale
3. Resizes it to 28 × 28
4. Inverts it when necessary
5. Normalizes the pixel values
6. Reshapes it for the CNN
7. Predicts the digit
8. Displays the prediction and confidence

Example filename: `digit5.png`

In [ ]:
def predict_custom_image(image_path):
    image = Image.open(image_path).convert("L")
    image = image.resize((28, 28))

    image_array = np.array(image)

    # Convert white-background/black-digit images
    # to the black-background/white-digit format used by MNIST
    if np.mean(image_array) > 127:
        image_array = 255 - image_array

    image_array = image_array.astype("float32") / 255.0
    image_array = image_array.reshape(1, 28, 28, 1)

    prediction = model.predict(image_array, verbose=0)

    predicted_digit = np.argmax(prediction[0])
    confidence = np.max(prediction[0]) * 100

    print("Predicted Digit:", predicted_digit)
    print(f"Confidence: {confidence:.2f}%")

    plt.figure(figsize=(3, 3))
    plt.imshow(image_array.reshape(28, 28), cmap="gray")
    plt.title(f"Predicted: {predicted_digit} ({confidence:.2f}%)")
    plt.axis("off")
    plt.show()

    return predicted_digit, confidence

# Example:
# predict_custom_image("digit5.png")

### Custom Image Testing

Run the following cell after placing your image in the notebook folder.

Change the filename to match your image.

```python
predict_custom_image("digit5.png")
```

For multiple images, run the function separately for each image.

## Results Summary

Record your actual results after running the notebook.

| Metric | Result |
|---|---|
| Training Images | 60,000 |
| Testing Images | 10,000 |
| Image Size | 28 × 28 |
| Number of Classes | 10 |
| Epochs | 10 |
| Test Accuracy | Enter actual result |
| Custom Image Predictions | Enter actual results |

## Conclusion

The CNN was successfully trained to classify handwritten digits using the MNIST dataset. The images were normalized, reshaped for CNN input, and the labels were one-hot encoded. The CNN used convolutional layers and max-pooling layers for feature extraction, dropout to reduce overfitting, and dense layers for classification.

The trained model was evaluated on the MNIST test dataset. The final test accuracy should be recorded from the actual output of the evaluation cell. The model was also prepared for testing custom handwritten digit images by resizing and normalizing them before prediction.

Overall, the experiment demonstrates that Convolutional Neural Networks are effective for handwritten digit classification.